# Walmart Sales Demand Forecasting

End-to-end ML walkthrough: EDA → Feature Engineering → Model Training → Evaluation

**Dataset**: [Kaggle – Walmart Recruiting: Store Sales Forecasting](https://www.kaggle.com/c/walmart-recruiting-store-sales-forecasting/data)

Place the four CSV files (`train.csv`, `test.csv`, `stores.csv`, `features.csv`) in the `../data/` directory before running.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from walmart_forecasting.data_loader import load_raw_data, preprocess
from walmart_forecasting.feature_engineering import build_features, ALL_FEATURES, TARGET
from walmart_forecasting.models import get_model
from walmart_forecasting.evaluation import evaluate, print_metrics

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load & Preprocess Data

In [ ]:
DATA_DIR = "../data/"

dfs = load_raw_data(DATA_DIR)
train_df, test_df = preprocess(dfs)

print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
train_df.head()

## 2. Exploratory Data Analysis

In [ ]:
print(train_df.describe())

In [ ]:
# Overall weekly sales distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train_df["Weekly_Sales"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Weekly Sales Distribution")
axes[0].set_xlabel("Weekly Sales ($)")
axes[0].set_ylabel("Count")

axes[1].hist(np.log1p(train_df["Weekly_Sales"]), bins=50, color="coral", edgecolor="white")
axes[1].set_title("Log(1+Weekly Sales) Distribution")
axes[1].set_xlabel("log(1 + Weekly Sales)")

plt.tight_layout()
plt.show()

In [ ]:
# Aggregate weekly sales over time
weekly = train_df.groupby("Date")["Weekly_Sales"].sum().reset_index()

plt.figure(figsize=(14, 4))
plt.plot(weekly["Date"], weekly["Weekly_Sales"], linewidth=0.8)
plt.title("Total Weekly Sales (all stores & depts)")
plt.xlabel("Date")
plt.ylabel("Total Weekly Sales ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Holiday vs non-holiday sales
fig, ax = plt.subplots(figsize=(7, 4))
train_df.groupby("IsHoliday")["Weekly_Sales"].median().plot(
    kind="bar", ax=ax, color=["steelblue", "coral"]
)
ax.set_xticklabels(["Non-Holiday", "Holiday"], rotation=0)
ax.set_title("Median Weekly Sales: Holiday vs Non-Holiday")
ax.set_ylabel("Median Weekly Sales ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Sales by store type
fig, ax = plt.subplots(figsize=(7, 4))
train_df.groupby("Type")["Weekly_Sales"].median().plot(
    kind="bar", ax=ax, color=["#2196F3", "#FF9800", "#4CAF50"]
)
ax.set_xticklabels(["Type A", "Type B", "Type C"], rotation=0)
ax.set_title("Median Weekly Sales by Store Type")
ax.set_ylabel("Median Weekly Sales ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric features
numeric_cols = ["Weekly_Sales", "Size", "Temperature", "Fuel_Price",
                "MarkDown1", "MarkDown2", "CPI", "Unemployment", "IsHoliday"]
corr = train_df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
train_feat = build_features(train_df, is_train=True)
print(f"Shape after feature engineering: {train_feat.shape}")
print(f"Features used: {[c for c in ALL_FEATURES if c in train_feat.columns]}")

## 4. Train / Validation Split

We use a **time-based split** to avoid data leakage: the last 20 % of weeks are held out for validation.

In [ ]:
SPLIT_FRAC = 0.8
cutoff = int(len(train_feat) * SPLIT_FRAC)

feature_cols = [c for c in ALL_FEATURES if c in train_feat.columns]

X_train = train_feat[feature_cols].iloc[:cutoff]
y_train = train_feat[TARGET].iloc[:cutoff]
X_val   = train_feat[feature_cols].iloc[cutoff:]
y_val   = train_feat[TARGET].iloc[cutoff:]
hol_val = train_feat["IsHoliday"].iloc[cutoff:].values

print(f"Train rows: {len(X_train):,}  |  Val rows: {len(X_val):,}")

## 5. Model Training & Evaluation

In [ ]:
results = {}

for model_name in ["linear_regression", "random_forest", "xgboost"]:
    print(f"Training {model_name} …", end=" ", flush=True)
    model = get_model(model_name)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    metrics = evaluate(y_val.values, preds, hol_val)
    results[model_name] = metrics
    print("done")
    print_metrics(metrics, model_name=model_name)

In [ ]:
# Compare WMAE across models
wmae_scores = {k: v["WMAE"] for k, v in results.items()}

plt.figure(figsize=(7, 4))
plt.bar(wmae_scores.keys(), wmae_scores.values(), color=["#2196F3", "#FF9800", "#4CAF50"])
plt.title("Validation WMAE by Model (lower is better)")
plt.ylabel("WMAE ($)")
plt.tight_layout()
plt.show()

## 6. Feature Importance (XGBoost)

In [ ]:
xgb_model = get_model("xgboost")
xgb_model.fit(X_train, y_train)
imp = xgb_model.feature_importances().head(20)

plt.figure(figsize=(10, 6))
imp.sort_values().plot(kind="barh", color="steelblue")
plt.title("Top 20 Feature Importances (XGBoost)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 7. Actual vs Predicted (Best Model)

In [ ]:
preds_xgb = xgb_model.predict(X_val)

# Plot for a single Store-Dept combination
val_df = train_feat.iloc[cutoff:].copy()
val_df["Predicted"] = preds_xgb

sample = val_df[(val_df["Store"] == 1) & (val_df["Dept"] == 1)].sort_values("Date")

plt.figure(figsize=(12, 4))
plt.plot(sample["Date"], sample[TARGET], label="Actual", linewidth=1.5)
plt.plot(sample["Date"], sample["Predicted"], label="Predicted", linewidth=1.5, linestyle="--")
plt.title("Actual vs Predicted Weekly Sales – Store 1, Dept 1")
plt.xlabel("Date")
plt.ylabel("Weekly Sales ($)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Save Best Model

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

# Retrain on full data and save
final_model = get_model("xgboost")
final_model.fit(train_feat[feature_cols], train_feat[TARGET])
final_model.save("../models/xgboost.pkl")
print("Best model saved to models/xgboost.pkl")